## 這裡要做訓練模型

In [7]:
# ------------------------------------
# 讀檔:取得csv的絕對路徑
# ------------------------------------

import os
#current_dir = os.path.dirname(os.path.abspath(__file__))
current_dir:str = os.getcwd()
csv_path:str = os.path.join(current_dir, "Salary_Data2.csv")

if not os.path.exists(csv_path):
    raise FileNotFoundError("找不到資料檔案")
print(csv_path)


C:\Users\User\Documents\GitHub\__2026_07_03__\backend\0731\Salary_Data2.csv


In [8]:
# -----------------------------------------------------
# 2. 建立並擬合 OrdianlEncoder (學歷高低 有分數之差)
# -----------------------------------------------------
import pandas as pd
import time
from pandas import DataFrame
from sklearn.preprocessing import OrdinalEncoder

data:DataFrame = pd.read_csv(csv_path)
#display(data)

#開始時間
star_time:float = time.time()

oe = OrdinalEncoder(categories=[['高中以下','大學','碩士以上']])
data['EducationLevel'] = oe.fit_transform(data[['EducationLevel']]) #把轉換的資料取代原本的學歷
#display(data)
display(data.head(5))

#要確認City裡面有多少個值
data['City'].unique()

,YearsExperience,EducationLevel,City,Salary
0,3.0,1.0,城市A,45.9
1,7.8,2.0,城市C,80.5
2,2.3,0.0,城市A,25.2
3,5.1,0.0,城市A,30.4
4,10.0,2.0,城市B,65.7


<StringArray>
['城市A', '城市C', '城市B']
Length: 3, dtype: str

In [9]:
# -----------------------------------------------------
# 3. 建立並擬合 OneHotEncoder (城市：城市A, 城市B, 城市C)
# -----------------------------------------------------
from sklearn.preprocessing import OneHotEncoder
#display(data['City'].unique()) #檢查是否有空值

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') #轉換器 轉為陣列
ohe.fit(pd.DataFrame([["城市A"], ["城市B"], ["城市C"]], columns=["City"])) #訓練:定義有哪些欄位值
city_encoded = ohe.transform(data[['City']]) #轉換成多欄數值的 Ndarry
city_cols = ohe.get_feature_names_out(['City'])  #建立欄位名稱
city_df = pd.DataFrame(city_encoded,columns=city_cols) #建立DataFrame
data = pd.concat([data,city_df],axis=1).drop('City',axis=1) #把資料串接上來 drop掉原本的
display(data)

,YearsExperience,EducationLevel,Salary,City_城市A,City_城市B,City_城市C
0,3.0,1.0,45.9,1.0,0.0,0.0
1,7.8,2.0,80.5,0.0,0.0,1.0
2,2.3,0.0,25.2,1.0,0.0,0.0
3,5.1,0.0,30.4,1.0,0.0,0.0
4,10.0,2.0,65.7,0.0,1.0,0.0
5,1.2,2.0,60.8,0.0,0.0,1.0
6,8.6,1.0,50.1,0.0,0.0,1.0
7,6.9,2.0,70.3,1.0,0.0,0.0
8,4.2,1.0,40.7,1.0,0.0,0.0
9,2.4,0.0,28.1,1.0,0.0,0.0


In [18]:
# -----------------------------------------------------
# 4. 開始訓練
# -----------------------------------------------------
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso,LinearRegression

# 定義特徵欄位與目標變數
feature_names = ['YearsExperience',	'EducationLevel','City_城市A','City_城市B','City_城市C']
X = data[feature_names]
y = data['Salary']

# 資料切割 訓練集跟測試集
test_size = 0.2
random_state = 76

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = test_size, random_state = random_state
)


# YearsExperience數字最大 影響權重特別大==>要做標準化
# train_要一邊訓練一邊記憶 用fit_transform
# test 只要轉換 transform
scaler = StandardScaler() #給他一個短名稱
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#設定模型
model_type = "LinearRegression "
alpha = 1.0
model_type_clean = model_type.strip()
if model_type_clean.lower() == "lasso":
    model = Lasso(alpha=alpha, random_state=random_state)
    actual_model_name = f"Lasso 迴歸(α={alpha})"
    model_type_clean="Lasso"
elif model_type_clean.lower() == "ridge":
    model = Ridge(alpha=alpha, random_state=random_state)
    actual_model_name = f"Ridge 嶺迴歸(α={alpha})"
    model_type_clean="Ridge"
else:
    model = LinearRegression()
    actual_model_name = "多元線性迴歸 (OLS)"
    model_type_clean="LinearRegression"


print(f'開始訓練{actual_model_name} 測試集比例:{test_size},隨機總子:{random_state})...')
model.fit(X_train_scaled,y_train) 
train_time = time.time() - star_time
display(train_time)

開始訓練多元線性迴歸 (OLS) 測試集比例:0.2,隨機總子:76)...


2660.72523188591

In [20]:
# -----------------------------------------------------
# 5. 取得模型的權重,偏移值,評估值R2
# -----------------------------------------------------

r2 = model.score(X_test_scaled, y_test)
coefs = model.coef_ #權重
intercept = model.intercept_
feature_coefs = {
    name:float (coef) for name, coef in zip(feature_names,coefs)
}
display(feature_coefs,f'R2:{r2}')

{'YearsExperience': 5.69635898409509,
 'EducationLevel': 15.307434019228705,
 'City_城市A': 0.5656436343757925,
 'City_城市B': -3.5283993727742757,
 'City_城市C': 1.3122634452625777}

'R2:0.8462535226367868'

In [21]:
# ----------------------------------------------------------
# 存模型以及所有相關預處理器與元數據 (Metadata)
# ----------------------------------------------------------
import joblib

model_data = {
    "model": model,
    "oe": oe,
    "ohe": ohe,
    "scaler": scaler,
    "r2": float(r2),
    "coef": [float(c) for c in coefs],
    "intercept": float(intercept),
    "feature_names": feature_names,
    "feature_coefs": feature_coefs,
    "model_type": model_type_clean,
    "alpha": float(alpha),
    "train_time": float(train_time),
    "test_size": test_size,
    "random_state": random_state
}

model_filename = os.path.join(current_dir, "salary_model.joblib")
print(f"正在將模型、預處理器與元數據序列化並儲存至 {model_filename}...")
joblib.dump(model_data,model_filename)
print("模型儲存成功！")

正在將模型、預處理器與元數據序列化並儲存至 C:\Users\User\Documents\GitHub\__2026_07_03__\backend\0731\salary_model.joblib...
模型儲存成功！
